In [1]:
import os
import gc
import sys
import time
import inspect
import statistics
from pathlib import Path

import topologicpy

from topologicpy.Core import Core
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Vertex import Vertex


# =============================================================================
# SETTINGS
# =============================================================================

RUNS = 3

WIDTH = 10
LENGTH = 10
HEIGHT = 10

U_SIDES = 10
V_SIDES = 10
W_SIDES = 10

EXPECTED_CELLS = 1000
EXPECTED_FACES = 3300
EXPECTED_EDGES = 3630
EXPECTED_VERTICES = 1331
EXPECTED_VOLUME = 1000.0

BACKENDS = [
    "pythonocc",
    "topologic_core",
]


# =============================================================================
# ENVIRONMENT
# =============================================================================

def print_environment_info():
    print("=" * 80)
    print("ENVIRONMENT")
    print("=" * 80)

    print("\nPython executable:")
    print(sys.executable)

    print("\nTopologicPy:")
    print(Path(topologicpy.__file__).resolve())

    try:
        import OCC

        print("\nPythonOCC:")
        print(Path(OCC.__file__).resolve())

    except Exception:
        print("\nPythonOCC:")
        print("NOT AVAILABLE")

    print()


# =============================================================================
# BACKEND
# =============================================================================

def select_backend(backend_name):
    """
    Selects and reports the requested backend.
    """

    os.environ["TOPOLOGICPY_CORE_BACKEND"] = backend_name
    Core.ResetBackend()

    backend = Core.Backend()
    backend_class = type(backend)

    print(f"Requested backend : {backend_name}")
    print(f"Active backend    : {backend_class.__name__}")
    print(f"Backend module    : {backend_class.__module__}")

    try:
        print(
            "Backend file      :",
            Path(
                inspect.getfile(backend_class)
            ).resolve()
        )
    except Exception:
        print("Backend file      : unavailable")

    return backend


# =============================================================================
# CELL COMPLEX
# =============================================================================

def create_cellcomplex():
    """
    Creates a regular 10 x 10 x 10 CellComplex containing 1000 cells.
    """

    return CellComplex.Prism(
        width=WIDTH,
        length=LENGTH,
        height=HEIGHT,
        uSides=U_SIDES,
        vSides=V_SIDES,
        wSides=W_SIDES,
    )


# =============================================================================
# QUERY TIMER
# =============================================================================

def timed_query(fn):
    """
    Executes a topology query and returns its result and elapsed time.
    """

    start = time.perf_counter()

    result = fn() or []

    elapsed = time.perf_counter() - start

    return result, elapsed


# =============================================================================
# VALIDATION
# =============================================================================

def validate_cellcomplex(cc):
    """
    Validates topology counts, bounding dimensions and aggregate volume.
    """

    if cc is None:
        return {
            "valid": False,
            "reason": "CellComplex is None",
        }

    # -------------------------------------------------------------------------
    # Individual topology extraction timings
    # -------------------------------------------------------------------------

    cells, cells_time = timed_query(
        lambda: Topology.Cells(
            cc,
            silent=True
        )
    )

    faces, faces_time = timed_query(
        lambda: Topology.Faces(
            cc,
            silent=True
        )
    )

    edges, edges_time = timed_query(
        lambda: Topology.Edges(
            cc,
            silent=True
        )
    )

    vertices, vertices_time = timed_query(
        lambda: Topology.Vertices(
            cc,
            silent=True
        )
    )

    extraction_time = (
        cells_time
        + faces_time
        + edges_time
        + vertices_time
    )

    # -------------------------------------------------------------------------
    # Bounding dimensions
    # -------------------------------------------------------------------------

    if vertices:

        xs = [
            Vertex.X(v)
            for v in vertices
        ]

        ys = [
            Vertex.Y(v)
            for v in vertices
        ]

        zs = [
            Vertex.Z(v)
            for v in vertices
        ]

        dx = max(xs) - min(xs)
        dy = max(ys) - min(ys)
        dz = max(zs) - min(zs)

    else:
        dx = 0.0
        dy = 0.0
        dz = 0.0

    # -------------------------------------------------------------------------
    # Volume
    # -------------------------------------------------------------------------

    start = time.perf_counter()

    total_volume = sum(
        Cell.Volume(cell)
        for cell in cells
    )

    volume_time = time.perf_counter() - start

    # -------------------------------------------------------------------------
    # Correctness
    # -------------------------------------------------------------------------

    valid = (
        len(cells) == EXPECTED_CELLS
        and len(faces) == EXPECTED_FACES
        and len(edges) == EXPECTED_EDGES
        and len(vertices) == EXPECTED_VERTICES
        and abs(dx - WIDTH) < 0.001
        and abs(dy - LENGTH) < 0.001
        and abs(dz - HEIGHT) < 0.001
        and abs(total_volume - EXPECTED_VOLUME) < 0.001
    )

    return {
        "valid": valid,

        "cells": len(cells),
        "faces": len(faces),
        "edges": len(edges),
        "vertices": len(vertices),

        "cells_time": cells_time,
        "faces_time": faces_time,
        "edges_time": edges_time,
        "vertices_time": vertices_time,

        "extraction_time": extraction_time,

        "dx": dx,
        "dy": dy,
        "dz": dz,

        "volume": total_volume,
        "volume_time": volume_time,
    }


# =============================================================================
# BENCHMARK ONE BACKEND
# =============================================================================

def benchmark_backend(
    backend_name,
    runs=3
):

    print("\n" + "=" * 80)
    print(
        f"BACKEND: {backend_name}"
    )
    print("=" * 80)

    backend = select_backend(
        backend_name
    )

    # -------------------------------------------------------------------------
    # Warm-up
    # -------------------------------------------------------------------------

    print("\nWarm-up...")

    warmup = CellComplex.Prism(
        width=1,
        length=1,
        height=1,
        uSides=2,
        vSides=2,
        wSides=2,
    )

    del warmup
    gc.collect()

    # -------------------------------------------------------------------------
    # Construction benchmark
    # -------------------------------------------------------------------------

    construction_times = []

    print(
        f"\nBenchmarking {runs} runs:\n"
    )

    final_cc = None

    for i in range(runs):

        gc.collect()

        start = time.perf_counter()

        cc = create_cellcomplex()

        elapsed = (
            time.perf_counter()
            - start
        )

        construction_times.append(
            elapsed
        )

        # Lightweight correctness check during timing runs.
        cells = (
            Topology.Cells(
                cc,
                silent=True
            )
            or []
        )

        valid_cell_count = (
            len(cells)
            == EXPECTED_CELLS
        )

        print(
            f"Run {i + 1}: "
            f"{elapsed:8.4f} sec | "
            f"Cells: {len(cells):4d} | "
            f"Valid: {valid_cell_count}"
        )

        if final_cc is not None:
            del final_cc

        final_cc = cc

        del cells
        gc.collect()

    # -------------------------------------------------------------------------
    # Statistics
    # -------------------------------------------------------------------------

    best = min(
        construction_times
    )

    median = statistics.median(
        construction_times
    )

    mean = statistics.mean(
        construction_times
    )

    print("\nConstruction timing:")
    print(
        f"Best   : {best:.4f} sec"
    )
    print(
        f"Median : {median:.4f} sec"
    )
    print(
        f"Mean   : {mean:.4f} sec"
    )

    # -------------------------------------------------------------------------
    # Full validation
    # -------------------------------------------------------------------------

    validation = validate_cellcomplex(
        final_cc
    )

    print("\nTopology validation:")

    print(
        f"Cells    : "
        f"{validation['cells']:5d} "
        f"(expected {EXPECTED_CELLS})"
    )

    print(
        f"Faces    : "
        f"{validation['faces']:5d} "
        f"(expected {EXPECTED_FACES})"
    )

    print(
        f"Edges    : "
        f"{validation['edges']:5d} "
        f"(expected {EXPECTED_EDGES})"
    )

    print(
        f"Vertices : "
        f"{validation['vertices']:5d} "
        f"(expected {EXPECTED_VERTICES})"
    )

    # -------------------------------------------------------------------------
    # Individual extraction timing
    # -------------------------------------------------------------------------

    print("\nTopology extraction timing:")

    print(
        f"Cells    : "
        f"{validation['cells_time']:.4f} sec"
    )

    print(
        f"Faces    : "
        f"{validation['faces_time']:.4f} sec"
    )

    print(
        f"Edges    : "
        f"{validation['edges_time']:.4f} sec"
    )

    print(
        f"Vertices : "
        f"{validation['vertices_time']:.4f} sec"
    )

    print(
        f"Total    : "
        f"{validation['extraction_time']:.4f} sec"
    )

    # -------------------------------------------------------------------------
    # Geometry
    # -------------------------------------------------------------------------

    print("\nGeometry validation:")

    print(
        f"X size   : "
        f"{validation['dx']:.6f}"
    )

    print(
        f"Y size   : "
        f"{validation['dy']:.6f}"
    )

    print(
        f"Z size   : "
        f"{validation['dz']:.6f}"
    )

    print(
        f"Volume   : "
        f"{validation['volume']:.6f}"
    )

    print(
        f"Volume calculation: "
        f"{validation['volume_time']:.4f} sec"
    )

    print(
        "\nVALID:",
        validation["valid"]
    )

    return {
        "requested_backend":
            backend_name,

        "active_backend":
            type(backend).__name__,

        "construction_times":
            construction_times,

        "best":
            best,

        "median":
            median,

        "mean":
            mean,

        "validation":
            validation,
    }


# =============================================================================
# RUN
# =============================================================================

print_environment_info()

results = {}

for backend_name in BACKENDS:

    results[backend_name] = (
        benchmark_backend(
            backend_name,
            runs=RUNS
        )
    )


# =============================================================================
# FINAL COMPARISON
# =============================================================================

occ = results["pythonocc"]
core = results["topologic_core"]

ov = occ["validation"]
cv = core["validation"]


print("\n" + "=" * 80)
print("FINAL COMPARISON")
print("=" * 80)


# -------------------------------------------------------------------------
# Construction
# -------------------------------------------------------------------------

print("\nCONSTRUCTION")

print(
    f"\n{'Backend':<25}"
    f"{'Best':>12}"
    f"{'Median':>12}"
    f"{'Mean':>12}"
)

print("-" * 61)

print(
    f"{occ['active_backend']:<25}"
    f"{occ['best']:>12.4f}"
    f"{occ['median']:>12.4f}"
    f"{occ['mean']:>12.4f}"
)

print(
    f"{core['active_backend']:<25}"
    f"{core['best']:>12.4f}"
    f"{core['median']:>12.4f}"
    f"{core['mean']:>12.4f}"
)


construction_ratio = (
    occ["median"]
    / core["median"]
)

print(
    "\nPythonOCC / TopologicCore "
    f"construction ratio: "
    f"{construction_ratio:.2f}x"
)


# -------------------------------------------------------------------------
# Extraction
# -------------------------------------------------------------------------

print("\nTOPOLOGY EXTRACTION")

print(
    f"\n{'Query':<15}"
    f"{'PythonOCC':>14}"
    f"{'TopologicCore':>16}"
    f"{'Ratio':>12}"
)

print("-" * 57)

for key, label in [
    ("cells_time", "Cells"),
    ("faces_time", "Faces"),
    ("edges_time", "Edges"),
    ("vertices_time", "Vertices"),
    ("extraction_time", "Total"),
]:

    occ_time = ov[key]
    core_time = cv[key]

    ratio = (
        occ_time / core_time
        if core_time > 0
        else float("inf")
    )

    print(
        f"{label:<15}"
        f"{occ_time:>14.4f}"
        f"{core_time:>16.4f}"
        f"{ratio:>12.2f}x"
    )


# -------------------------------------------------------------------------
# Volume
# -------------------------------------------------------------------------

volume_ratio = (
    ov["volume_time"]
    / cv["volume_time"]
)

print("\nVOLUME CALCULATION")

print(
    f"PythonOCC    : "
    f"{ov['volume_time']:.4f} sec"
)

print(
    f"TopologicCore: "
    f"{cv['volume_time']:.4f} sec"
)

print(
    f"Ratio        : "
    f"{volume_ratio:.2f}x"
)


# -------------------------------------------------------------------------
# End-to-end
# -------------------------------------------------------------------------

occ_total = (
    occ["median"]
    + ov["extraction_time"]
    + ov["volume_time"]
)

core_total = (
    core["median"]
    + cv["extraction_time"]
    + cv["volume_time"]
)

end_to_end_ratio = (
    occ_total
    / core_total
)

print("\nEND-TO-END")

print(
    f"PythonOCC    : "
    f"{occ_total:.4f} sec"
)

print(
    f"TopologicCore: "
    f"{core_total:.4f} sec"
)

print(
    f"Ratio        : "
    f"{end_to_end_ratio:.2f}x"
)


# -------------------------------------------------------------------------
# Correctness equivalence
# -------------------------------------------------------------------------

same_topology = (
    ov["cells"] == cv["cells"]
    and ov["faces"] == cv["faces"]
    and ov["edges"] == cv["edges"]
    and ov["vertices"] == cv["vertices"]
)

same_geometry = (
    abs(ov["dx"] - cv["dx"]) < 0.001
    and abs(ov["dy"] - cv["dy"]) < 0.001
    and abs(ov["dz"] - cv["dz"]) < 0.001
    and abs(
        ov["volume"]
        - cv["volume"]
    ) < 0.001
)

print("\nEQUIVALENCE")

print(
    "Same topology :",
    same_topology
)

print(
    "Same geometry :",
    same_geometry
)

print(
    "PythonOCC valid:",
    ov["valid"]
)

print(
    "TopologicCore valid:",
    cv["valid"]
)


# =============================================================================
# RESTORE DEFAULT
# =============================================================================

os.environ.pop(
    "TOPOLOGICPY_CORE_BACKEND",
    None
)

Core.ResetBackend()

print("\n" + "=" * 80)
print("DEFAULT RESTORED")
print("=" * 80)

print(
    "Active backend:",
    type(Core.Backend()).__name__
)

ENVIRONMENT

Python executable:
c:\Users\sarwj\AppData\Local\miniconda3\envs\occ-test\python.exe

TopologicPy:
C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\src\topologicpy\__init__.py

PythonOCC:
C:\Users\sarwj\AppData\Local\miniconda3\envs\occ-test\Lib\site-packages\OCC\__init__.py


BACKEND: pythonocc
Requested backend : pythonocc
Active backend    : PythonOCCBackend
Backend module    : topologicpy.pythonocc_backend.backend
Backend file      : C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\src\topologicpy\pythonocc_backend\backend.py

Warm-up...

Benchmarking 3 runs:

Run 1:   1.1280 sec | Cells: 1000 | Valid: True
Run 2:   1.0486 sec | Cells: 1000 | Valid: True
Run 3:   1.0280 sec | Cells: 1000 | Valid: True

Construction timing:
Best   : 1.0280 sec
Median : 1.0486 sec
Mean   : 1.0682 sec

Topology validation:
Cells    :  1000 (expected 1000)
Faces    :  3300 (expected 3300)
Edges    :  3630 (expected 3630)
Vertices :  1331 (ex